# Smoke detector v2 (yolo11n) on dataset_v2 - train on Colab

ชุดข้อมูลสร้างด้วย `tools/build_smoke_dataset_v2.py` = dataset v1 (ScPuteri CC BY 4.0 + ITLP CC BY 4.0) + Indoor Fire Smoke (Putra, A.K., Binus University, Zenodo DOI 10.5281/zenodo.15826133, CC BY 4.0, จาก Roboflow indoor-fire-smoke) class เดียวคือ smoke

**กฎ (ห้ามเปลี่ยน)**
- ใช้แค่ `dataset_v2.zip` ที่สร้างจากสคริปต์ ห้ามใส่ภาพหรือคลิปจาก `~/smoke_eval_clips` หรือฉากทดสอบสด
- train/val แบ่งไว้แล้วตามกลุ่มภาพ ห้ามแบ่งใหม่ที่นี่
- ผล val ที่นี่**ไม่ใช่**ผลวัดของโปรเจกต์ ผลจริงต้องวัดด้วย `tools/eval_smoke_clips.py` และ `tools/smoke_live_test.py` บน Mac

**ก่อนรัน:** Runtime → Change runtime type → **T4 GPU** เวลาโดยประมาณ: 50 epoch ประมาณ 10–20 นาที


In [ ]:
# 1. same Ultralytics as the Mac (8.4.136) and check the GPU
!pip install -q ultralytics==8.4.136
!nvidia-smi --query-gpu=name,memory.total --format=csv
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert ultralytics.__version__ == "8.4.136", "use the same version as the Mac"


In [ ]:
# 2. dataset.zip from Google Drive -> local disk, then check it
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import zipfile, shutil, json

ZIP = Path('/content/drive/MyDrive/smoke_custom/dataset_v2.zip')
OUT_DIR = Path('/content/drive/MyDrive/smoke_custom/smoke_custom_yolo11n_v2')
WORK = Path('/content/smoke_custom_v2')
assert 'smoke_eval_clips' not in str(ZIP), 'test clips must never be used for training'
assert ZIP.is_file(), f'{ZIP} not found'
if WORK.exists():
    shutil.rmtree(WORK)
with zipfile.ZipFile(ZIP) as z:
    bad = [n for n in z.namelist() if 'smoke_eval_clips' in n]
    assert not bad, f'zip contains test clips: {bad[:3]}'
    z.extractall(WORK)
DATA = next(WORK.rglob('data.yaml'))
ROOT = DATA.parent
print('data.yaml:', DATA, '\n' + DATA.read_text())
info = json.loads((ROOT / 'build_info.json').read_text())
print(json.dumps(info['report'], indent=2))
for split in ('train', 'val'):
    n_img = len(list((ROOT / split / 'images').iterdir())); n_lab = len(list((ROOT / split / 'labels').iterdir()))
    print(f'{split}: images {n_img} labels {n_lab}')
    assert n_img == n_lab == info['report'][split]['images'], 'dataset does not match build_info.json'
assert info['report']['near_duplicates_val_vs_train'] == 0


In [ ]:
# 3. train yolo11n, smoke only (fixed settings)
from ultralytics import YOLO
ARGS = dict(data=str(DATA), epochs=50, imgsz=640, batch=32, seed=0, deterministic=True,
            patience=50, project='/content/runs', name='smoke_custom_yolo11n_v2', exist_ok=True)
model = YOLO('yolo11n.pt')   # COCO-pretrained yolo11n from Ultralytics
model.train(**ARGS)


In [ ]:
# 4. save weights + record to Drive (download both to models/smoke/ on the Mac)
import datetime, hashlib
best = Path('/content/runs/smoke_custom_yolo11n_v2/weights/best.pt')
m = YOLO(str(best))
val = m.val(data=str(DATA), split='val', imgsz=640)
OUT_DIR.mkdir(parents=True, exist_ok=True)
out = OUT_DIR / 'smoke_custom_yolo11n_v2.pt'
shutil.copy2(best, out)
record = {
    'file': out.name, 'sha256': hashlib.sha256(out.read_bytes()).hexdigest(),
    'date': datetime.datetime.now().isoformat(timespec='minutes'), 'base': 'yolo11n.pt',
    'classes': m.names, 'dataset_build_info': info,
    'train_args': {k: v for k, v in ARGS.items() if k != 'data'},
    'val_split_metrics': {'mAP50': float(val.box.map50), 'mAP50-95': float(val.box.map),
                          'precision': float(val.box.mp), 'recall': float(val.box.mr)},
    'ultralytics': ultralytics.__version__, 'torch': torch.__version__,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    'note': 'val metrics are on the dataset split, not on our camera; measure with eval_smoke_clips / smoke_live_test',
}
(OUT_DIR / 'train_info.json').write_text(json.dumps(record, indent=2, ensure_ascii=False, default=str))
print(json.dumps(record['val_split_metrics'], indent=2), '\nsha256', record['sha256'])


## ขั้นต่อไปบน Mac

```bash
~/Desktop/nursery_guard_ai/venv/bin/python tools/smoke_live_test.py --model models/smoke/smoke_custom_yolo11n_v2.pt
```
เกณฑ์ A/B/C และ conf 0.25 เดิม ห้ามแก้ ห้ามต่อเข้า pipeline หลักจนกว่าจะสั่ง
